In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,RandomizedSearchCV 
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.neighbors import KNeighborsRegressor  
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.svm import SVR 
from xgboost import XGBRegressor 
from catboost import CatBoostRegressor

from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score 

import warnings 
warnings.filterwarnings('ignore') 

In [2]:
df = pd.read_csv('data/StudentsPerformance.csv')
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
df.shape

(1000, 8)

In [4]:
X = df.drop('math score',axis=1)
y = df['math score']

In [5]:
X.shape,y.shape

((1000, 7), (1000,))

In [6]:
X.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [7]:
y.head()

0    72
1    69
2    90
3    47
4    76
Name: math score, dtype: int64

In [8]:
df.columns

Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course', 'math score', 'reading score',
       'writing score'],
      dtype='object')

In [9]:
print("Categories in all the categorical features in the dataset :")
print('\n')
print("Gender :",df['gender'].unique())
print('\n')
print("Race/Ethnicity :",df['race/ethnicity'].unique())
print('\n')
print("Parental Level of Education :",df['parental level of education'].unique())
print('\n')
print("Lunch: ",df['lunch'].unique())
print('\n')
print('Test Preparation Course : ',df['test preparation course'].unique())

Categories in all the categorical features in the dataset :


Gender : ['female' 'male']


Race/Ethnicity : ['group B' 'group C' 'group A' 'group D' 'group E']


Parental Level of Education : ["bachelor's degree" 'some college' "master's degree" "associate's degree"
 'high school' 'some high school']


Lunch:  ['standard' 'free/reduced']


Test Preparation Course :  ['none' 'completed']


In [10]:
numerical_features = X.select_dtypes(exclude='object').columns
categorical_features = X.select_dtypes(include='object').columns

In [11]:
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

In [12]:
preprocessor = ColumnTransformer(
    [
        ("ONE_HOT_ENCODER",categorical_transformer,categorical_features),
        ("SCALER",numerical_transformer,numerical_features)
    ]
)

In [13]:
preprocessor

ColumnTransformer(transformers=[('ONE_HOT_ENCODER', OneHotEncoder(),
                                 Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course'],
      dtype='object')),
                                ('SCALER', StandardScaler(),
                                 Index(['reading score', 'writing score'], dtype='object'))])

In [14]:
X = preprocessor.fit_transform(X)

In [15]:
X.shape

(1000, 19)

In [17]:
X

array([[ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.19399858,  0.39149181],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.42747598,  1.31326868],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.77010859,  1.64247471],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.12547206, -0.20107904],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.60515772,  0.58901542],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.15336989,  1.18158627]])

In [19]:
X_df = pd.DataFrame(X,columns=preprocessor.get_feature_names_out())
X_df

,ONE_HOT_ENCODER__gender_female,ONE_HOT_ENCODER__gender_male,ONE_HOT_ENCODER__race/ethnicity_group A,ONE_HOT_ENCODER__race/ethnicity_group B,ONE_HOT_ENCODER__race/ethnicity_group C,ONE_HOT_ENCODER__race/ethnicity_group D,ONE_HOT_ENCODER__race/ethnicity_group E,ONE_HOT_ENCODER__parental level of education_associate's degree,ONE_HOT_ENCODER__parental level of education_bachelor's degree,ONE_HOT_ENCODER__parental level of education_high school,ONE_HOT_ENCODER__parental level of education_master's degree,ONE_HOT_ENCODER__parental level of education_some college,ONE_HOT_ENCODER__parental level of education_some high school,ONE_HOT_ENCODER__lunch_free/reduced,ONE_HOT_ENCODER__lunch_standard,ONE_HOT_ENCODER__test preparation course_completed,ONE_HOT_ENCODER__test preparation course_none,SCALER__reading score,SCALER__writing score
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.193999,0.391492
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.427476,1.313269
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.770109,1.642475
3,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.833899,-1.583744
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.605158,0.457333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,2.044215,1.774157
996,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.970952,-0.859491
997,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.125472,-0.201079
998,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.605158,0.589015


#### Train_Test_Split

In [20]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [22]:
X_train.shape,X_test.shape

((800, 19), (200, 19))

#### Create Evaluate Function

In [24]:
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = np.sqrt(mse)
    r2_score = r2_score(true,predicted)

    return mae,mse,rmse,r2_score

In [25]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "LASSO": Lasso(),
    "ElasticNet": ElasticNet(),
    "K-Neigbors Regressor": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "XGB Regressor": XGBRegressor(),
    "CatBoost Regressor": CatBoostRegressor()
}

In [28]:
print(len(models))
print(models.values())
print('\n')
print(models.keys())

11
dict_values([LinearRegression(), Ridge(), Lasso(), ElasticNet(), KNeighborsRegressor(), SVR(), DecisionTreeRegressor(), RandomForestRegressor(), AdaBoostRegressor(), XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...), CatBoostRegressor(loss_function='RMSE')])


dict_keys(['Linear Regression', 'Ridge'

In [ ]:
models_list = []
r2_list = []